# DataWizard — walkthrough

An AI agent that analyzes CSV files through **LangChain tools**. Runs **offline by default** (rule-based stand-in model: no API key, no cost). Set `PROVIDER = "gemini"` (free key) or `"openai"` for a real LLM.

In [1]:
PROVIDER = "offline"  # "gemini" | "openai" | "offline"

import os, json
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from data_agent import (TOOLS, set_data_dir, list_csv_files, get_dataset_summaries, call_dataframe_method,
                        evaluate_classification_dataset, evaluate_regression_dataset,
                        build_agent, get_llm, ask, final_answer, tool_calls)
from data_agent.cli import print_trace
set_data_dir(Path.cwd().parent / "data" if Path.cwd().name == "notebooks" else "data")

## 1. The tools — what the LLM sees
Name + docstring + argument schema. The model picks tools from these descriptions alone.

In [2]:
for t in TOOLS:
    print(f"{t.name:32} args={list(t.args)}")

list_csv_files                   args=[]
preload_datasets                 args=['paths']
get_dataset_summaries            args=['dataset_paths']
call_dataframe_method            args=['file_name', 'method']
evaluate_classification_dataset  args=['file_name', 'target_column']
evaluate_regression_dataset      args=['file_name', 'target_column']


## 2. Call the tools directly (no LLM)
The agent passes **file names**, not data — the DataFrames stay cached in memory.

In [3]:
print(list_csv_files.invoke({}))
summary = get_dataset_summaries.invoke({"dataset_paths": ["customer-churn.csv"]})[0]
for c in summary["columns"]:
    print(f"{c['name']:18} {c['dtype']:8} n_unique={c['n_unique']}")

['customer-churn.csv', 'house-prices.csv']
tenure_months      int64    n_unique=71
contract           str      n_unique=3
monthly_charges    float64  n_unique=743
total_charges      float64  n_unique=798
internet_service   str      n_unique=3
support_calls      int64    n_unique=9
paperless_billing  str      n_unique=2
churned            str      n_unique=2


In [4]:
print(call_dataframe_method.invoke({"file_name": "house-prices.csv", "method": "head"}))

   area_sqft  bedrooms  bathrooms  age_years  distance_to_center_km neighborhood garage  price_k
0     1737.0         4          2       32.0                    3.9     downtown    yes    420.7
1     1132.0         2          1       50.0                   19.5        north    yes    276.4
2     1938.0         5          3       15.0                    1.5         east    yes    411.5
3     2023.0         3          2       26.0                   12.5         east    yes    396.1
4      722.0         1          1       31.0                    5.0        north    yes    186.6


In [5]:
print(json.dumps(evaluate_classification_dataset.invoke({"file_name": "customer-churn.csv", "target_column": "churned"}), indent=1))
print(json.dumps(evaluate_regression_dataset.invoke({"file_name": "house-prices.csv", "target_column": "price_k"}), indent=1))

{
 "task": "classification",
 "target_column": "churned",
 "model": "RandomForestClassifier",
 "accuracy": 0.7438,
 "f1_macro": 0.7002,
 "baseline_accuracy": 0.6687,
 "classes": [
  "no",
  "yes"
 ],
 "n_train": 640,
 "n_test": 160,
 "top_features": [
  "tenure_months",
  "total_charges",
  "monthly_charges"
 ]
}


{
 "task": "regression",
 "target_column": "price_k",
 "model": "RandomForestRegressor",
 "r2_score": 0.8775,
 "mean_squared_error": 786.4592,
 "mean_absolute_error": 21.5911,
 "n_train": 470,
 "n_test": 118,
 "top_features": [
  "area_sqft",
  "age_years",
  "distance_to_center_km"
 ]
}


## 3. One step vs. the full loop
A model on its own decides **one** action. The agent (the lab's `AgentExecutor`, here `create_agent`) keeps looping — act, observe, decide — until it can answer. `stream` shows each step.

In [6]:
agent = build_agent(get_llm(PROVIDER))
for step in agent.stream({"messages": [("human", "Tell me about the datasets")]}, stream_mode="updates"):
    for node, update in step.items():
        msg = update["messages"][-1]
        what = [c["name"] for c in msg.tool_calls] if getattr(msg, "tool_calls", None) else str(msg.content)[:90]
        print(f"{node:6} → {what}")

model  → ['list_csv_files']
tools  → ["customer-churn.csv", "house-prices.csv"]
model  → ['get_dataset_summaries']
tools  → [{"file_name": "customer-churn.csv", "rows": 800, "columns": [{"name": "tenure_months", "d
model  → customer-churn.csv: 800 rows, 8 columns - tenure_months (int64), contract (str), monthly_c


## 4. The DataWizard question: classification or regression?

In [7]:
response = ask(agent, "Is each dataset a classification or a regression problem? Train and evaluate a model for each.")
print_trace(response["messages"])

[1] Human       Is each dataset a classification or a regression problem? Train and evaluate a model for each.
[2] AI → tool   list_csv_files({})
[3] Tool        list_csv_files → ["customer-churn.csv", "house-prices.csv"]
[4] AI → tool   get_dataset_summaries({'dataset_paths': ['customer-churn.csv', 'house-prices.csv']})
[5] Tool        get_dataset_summaries → [{"file_name": "customer-churn.csv", "rows": 800, "columns": [{"name": "tenure_months", "dtype": "int64", "n_unique": 71, "missing": 0}, {"name": "contract", "dtype": "str", "n_unique": 3, "missing": 0}, {"name": "monthly_charges", "dtype": "float64", "n_unique": 743, "missing": 0}, {"name": "total_charges", "dtype": "float64", "n_unique": 798, "missing": 0}, {"name": "internet_service", "dtype": "str", "n_unique": 3, "missing": 0}, {"name": "support_calls", "dtype": "int64", "n_unique": 9, "missing": 0}, {"name": "paperless_billing", "dtype": "str", "n_unique": 2, "missing": 0}, {"name": "chu … (+696 chars)
[6] AI → tool   evalu

## 5. Evaluate the agent: check the tool calls, not the wording

In [8]:
expected = {("evaluate_classification_dataset", "customer-churn.csv", "churned"),
            ("evaluate_regression_dataset", "house-prices.csv", "price_k")}
actual = {(c["name"], c["args"].get("file_name"), c["args"].get("target_column")) for c in tool_calls(response)}
for item in sorted(expected):
    print("✅" if item in actual else "❌", item)
print(f"\n{len(expected & actual)}/{len(expected)} checks passed")

✅ ('evaluate_classification_dataset', 'customer-churn.csv', 'churned')
✅ ('evaluate_regression_dataset', 'house-prices.csv', 'price_k')

2/2 checks passed
